# Video RAG with Weaviate & Google Gemini

This notebook demonstrates semantic search and RAG over video content with:
- A local MP4 video file (`video.mp4`).
- **ffmpeg-python** to split the video into overlapping chunks.
- **Weaviate** with `multi2vec_google_gemini` vectorizer to embed and index each clip using Gemini Embedding 2, Google’s first fully multimodal embedding model.
- **Gemini 3 Flash** to generate a response based on the relevant video segments.

Note: Place your `video.mp4` in the same directory as this notebook. You can use any MP4 file - we used the [Weaviate Agent Skills introduction video](https://www.youtube.com/watch?v=T0GD9xWihPI).

In [ ]:
# Setup and install necessary dependencies
!pip install -q weaviate-client ffmpeg-python google-genai

### Step 1: Connect to Weaviate

First, set up a Weaviate instance and connect to it. If you don’t already have one, sign up for a free Weaviate Cloud sandbox cluster [here](https://console.weaviate.cloud/signin?utm_source=github&utm_campaign=gemini_recipe).

Once your cluster is ready:

1. Copy your cluster URL and generate an API key.
2. Add them as environment variables named `WEAVIATE_URL` and `WEAVIATE_API_KEY`.
3. Go to [Google AI Studio](https://aistudio.google.com/api-keys), generate a Google API key, and save it as `GOOGLE_API_KEY` environment variable.


In [ ]:
import os
import base64
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.config import Configure, Property, DataType

WEAVIATE_URL = os.getenv("WEAVIATE_URL")
WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
    headers={
        "X-Goog-Api-Key": GOOGLE_API_KEY,
    }
)

print(f"Connected to Weaviate: {client.is_ready()}")

### Step 2: Create Weaviate Collection

In [ ]:
COLLECTION_NAME = "VideoSnippets"

if client.collections.exists(COLLECTION_NAME):
    print(f"Collection {COLLECTION_NAME} already exists.")
    # Un-comment the following lines to delete the existing collection
    # client.collections.delete(COLLECTION_NAME)
    # print(f"Deleted collection: {COLLECTION_NAME}")
else:
    collection = client.collections.create(
        name=COLLECTION_NAME,
        properties=[
            Property(name="video_clip", data_type=DataType.BLOB),  # base64 MP4 chunk
            Property(name="chunk_index", data_type=DataType.INT),
            Property(name="start_time", data_type=DataType.NUMBER),
            Property(name="end_time", data_type=DataType.NUMBER),
            Property(name="start_label", data_type=DataType.TEXT),
        ],
        vector_config=[
            Configure.Vectors.multi2vec_google_gemini(
                name="video_vector",
                video_fields=["video_clip"],
                model="gemini-embedding-2-preview",
                vector_index_config=Configure.VectorIndex.flat(),
            )
        ],
    )
    print(f"Created collection: {COLLECTION_NAME}")

### Step 3: Load Local Video

Place your `video.mp4` file in the same directory as this notebook before running this cell.  
You can use any MP4 file of your choice.

In [ ]:
VIDEO_PATH = "video.mp4"

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(
        f"'{VIDEO_PATH}' not found. Place your MP4 file in this directory."
    )

print(f"Using video: {VIDEO_PATH}")

### Step 4: Split Video into Overlapping Chunks

| Parameter | Value |
|-----------|-------|
| Chunk duration | 15 s |
| Overlap | 5 s |
| Step | 10 s |

Each chunk is written to `video_chunks/` directory.

In [ ]:
import ffmpeg

CHUNK_DURATION = 15  # seconds
OVERLAP = 5
STEP = CHUNK_DURATION - OVERLAP
CHUNKS_DIR = "video_chunks"

os.makedirs(CHUNKS_DIR, exist_ok=True)


def seconds_to_label(t: float) -> str:
    m, s = divmod(int(t), 60)
    return f"{m}:{s:02d}"


def video_to_base64(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def get_duration(path: str) -> float:
    probe = ffmpeg.probe(path)
    return float(probe["format"]["duration"])


def extract_chunk(
    input_path: str, output_path: str, start: float, duration: float
) -> None:
    """Cut a video segment (video + audio) starting at `start` for `duration` seconds."""
    (
        ffmpeg.input(input_path, ss=start, t=duration)
        .output(output_path, vcodec="libx264", acodec="aac", loglevel="error")
        .run(overwrite_output=True, quiet=True)
    )


chunks = []
total_duration = get_duration(VIDEO_PATH)
print(f"Video duration: {total_duration:.1f}s")

start = 0.0
idx = 0

while start < total_duration:
    end = min(start + CHUNK_DURATION, total_duration)
    seg_dur = end - start

    if seg_dur < 3:  # skip tiny trailing clips
        break

    clip_path = os.path.join(CHUNKS_DIR, f"chunk_{idx:04d}.mp4")
    extract_chunk(VIDEO_PATH, clip_path, start, seg_dur)

    chunks.append(
        {
            "chunk_index": idx,
            "start_time": start,
            "end_time": end,
            "start_label": seconds_to_label(start),
            "clip_path": clip_path,
        }
    )
    print(
        f"  Chunk {idx:03d}: {seconds_to_label(start)} – {seconds_to_label(end)}  →  {clip_path}"
    )
    idx += 1
    start += STEP

print(f"\nTotal chunks: {len(chunks)}")

### Step 5: Ingest Video Chunks into Weaviate

We base64-encode each MP4 clip and store it in the `video_clip` BLOB field. Weaviate forwards the raw clip bytes to Gemini's embedding API and stores the resulting vectors.

In [ ]:
collection = client.collections.get(COLLECTION_NAME)

In [ ]:
print("Starting ingestion...")
with collection.batch.dynamic() as batch:
    for chunk in chunks:
        batch.add_object(
            properties={
                "video_clip": video_to_base64(chunk["clip_path"]),
                "chunk_index": chunk["chunk_index"],
                "start_time": chunk["start_time"],
                "end_time": chunk["end_time"],
                "start_label": chunk["start_label"],
            }
        )
        print(
            f"  Indexed chunk {chunk['chunk_index']:03d} ({chunk['start_label']} – {seconds_to_label(chunk['end_time'])})"
        )

# Check for ingestion errors
if len(collection.batch.failed_objects) > 0:
    print("Some/All objects failed to import.")
else:
    print(f"\nSuccessfully indexed all {len(chunks)} chunks!")

### Step 6: Semantic Search over Video Chunks

In [ ]:
query = "How to install Weaviate skills?"

print(f"\nQuery: '{query}'")
print("-" * 50)

results = collection.query.near_text(
    query=query,
    limit=3,
    target_vector="video_vector",
    return_properties=["chunk_index", "start_time", "end_time", "start_label"],
)

for obj in results.objects:
    p = obj.properties
    print(
        f"  Chunk {p['chunk_index']:03d} | "
        f"{p['start_label']} – {seconds_to_label(p['end_time'])}"
    )

### Step 7: RAG - Answer a Question from Video Context

In [ ]:
from google import genai
from google.genai import types

gemini = genai.Client(api_key=GOOGLE_API_KEY)


def rag_answer(question: str, limit: int = 5) -> str:
    """
    1. Weaviate: semantic search over video embeddings → top-k chunks
    2. Gemini:   generate an answer using the actual video bytes inline
    """
    results = collection.query.near_text(
        query=question,
        limit=limit,
        target_vector="video_vector",
        return_properties=["video_clip", "chunk_index", "start_label", "end_time"],
    )

    # Build a multimodal prompt: one video part per retrieved chunk + the question
    parts = []
    for obj in results.objects:
        p = obj.properties
        video_bytes = base64.b64decode(p["video_clip"])
        parts.append(
            types.Part(
                inline_data=types.Blob(data=video_bytes, mime_type="video/mp4"),
            )
        )
        print(
            f"  Attaching chunk {p['chunk_index']:03d} ({p['start_label']} – {seconds_to_label(p['end_time'])})"
        )

    parts.append(types.Part(text=question))

    response = gemini.models.generate_content(
        model="gemini-3-flash-preview",
        contents=types.Content(parts=parts),
    )
    return response.text


question = "How to install Weaviate skill in Claude Code?"
print(f"Question: {question}\n")
answer = rag_answer(question)
print(f"\nAnswer:\n{answer}")

In [ ]:
client.close()